In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
import statsmodels.api as sm
import seaborn as sns
import bambi as bmb
import arviz as az
import scoringrules as sr

pa_gs = pd.read_csv('data/pa_data.csv')
temps = pd.read_csv("data/pa_temps_supplement.csv")
#combining main csv dataset with supplementary temperature data csv
pa_gs = pa_gs.merge(temps, on=["station_id", "year"], how="left")

#limiting dataset to just time interval of interest
pa_gs = pa_gs[pa_gs['year'] >= 1960]

#splitting dataframe in preparation for lagging of variable data
pa_gs_yr_gsl = pa_gs[['year', 'growing_season_length', 'last_spring_frost_doy', 'first_fall_frost_doy', 'station_id']]
pa_gs_non_gsl = pa_gs.drop(['growing_season_length', 'last_spring_frost_doy', 'first_fall_frost_doy'], axis = 1)

#creating lagged dataset where variable data is paired with GSL
pa_gs_lagged = pa_gs_non_gsl.copy(deep=True)
pa_gs_lagged['year'] += 1
pa_gs_lagged_vars = pa_gs_lagged.merge(pa_gs_yr_gsl, on=["station_id", "year"], how = "left")

In [3]:
def remove_nulls_simple(data_subset, variables):
        # Drop rows where any of the specified variable columns have nulls
        data_subset_cleaned = data_subset.dropna(subset=variables, ignore_index=True)
        # Subsets the cleaned dataframe back into only chosen variables
        vars_cleaned = data_subset_cleaned[variables]
        return vars_cleaned

def list_to_str(list_name):
    string = " + ".join(list_name)
    return string
    
def bayesian_model_unstandardized(data, target_var_name, cov_list):

    cov_str = list_to_str(cov_list)
    cov_list_target_var = list(cov_list) + [target_var_name] + ['year']
    data = remove_nulls_simple(data, cov_list_target_var)
    
    formula = bmb.Formula(f"{target_var_name} ~ {cov_str} + year",
                      f"sigma ~ {cov_str} + year")
    model = bmb.Model(formula, data=data, dropna = True)
    output = model.fit(idata_kwargs={'log_likelihood':True})

    resid = data[target_var_name] - model.predict(output, inplace=False).posterior["mu"].mean(("chain", "draw")).values
    
    return model, output, resid


all_covs_list = ['dtr_annual', 'dtr_spring', 'tmean_spring','tmean_fall','latitude','longitude','tmax_annual','oni_annual',
                'nao_annual','pna_annual','amo_annual','sst_north_atlantic','pwat_station','dewpoint_station',
                'soil_moisture_station','cloud_cover_station','evaporation_station',
                'sst_gulf_mexico','pwat_southeast_us','dewpoint_2m_southeast_us','soil_moisture_southeast_us',
                'cloud_cover_southeast_us','evaporation_southeast_us']

In [4]:
lagged_model, lagged_output, lagged_resid = bayesian_model_unstandardized(pa_gs_lagged_vars, 'growing_season_length', all_covs_list)

                                                            Grad                                                  
  Progress               Draw        Divergen…   Step size   evals       Speed                Elapsed    Remaini…  
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.148       31          119.95 draws/s       0:00:16    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.165       31          107.32 draws/s       0:00:18    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.165       31          123.96 draws/s       0:00:16    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.154       31          89.90 draws/s        0:00:22    0:00:00

/home/reu/.venv/lib/python3.14/site-packages/pymc/sampling/mcmc.py:1163: FutureWarning: Passing `log_likelihood` via `idata_kwargs` is deprecated and will be removed in future versions. Call `pm.compute_log_likelihood(idata)` instead.
  return _sample_return(
Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 29 seconds.


In [5]:
dtr_list = ["dtr_annual"]
lagged_model_dtr_only, lagged_output_dtr_only, lagged_resid_dtr_only = bayesian_model_unstandardized(pa_gs_lagged_vars, 'growing_season_length', dtr_list)

                                                            Grad                                                  
  Progress               Draw        Divergen…   Step size   evals       Speed                Elapsed    Remaini…  
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  ━━━━━━━━━━━━━━━━━━━━   2000        0           1.124       7           1113.91 draws/s      0:00:01    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.971       3           859.26 draws/s       0:00:02    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.868       3           929.26 draws/s       0:00:02    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.823       7           997.67 draws/s       0:00:02    0:00:00

/home/reu/.venv/lib/python3.14/site-packages/pymc/sampling/mcmc.py:1163: FutureWarning: Passing `log_likelihood` via `idata_kwargs` is deprecated and will be removed in future versions. Call `pm.compute_log_likelihood(idata)` instead.
  return _sample_return(
Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 4 seconds.


In [6]:
lagged_lsf_model, lagged_lsf_output, lagged_lsf_resid = bayesian_model_unstandardized(pa_gs_lagged_vars, 'last_spring_frost_doy', all_covs_list)

                                                            Grad                                                  
  Progress               Draw        Divergen…   Step size   evals       Speed                Elapsed    Remaini…  
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.153       31          139.69 draws/s       0:00:14    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.134       31          145.81 draws/s       0:00:13    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.132       31          126.94 draws/s       0:00:15    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.174       31          147.44 draws/s       0:00:13    0:00:00

/home/reu/.venv/lib/python3.14/site-packages/pymc/sampling/mcmc.py:1163: FutureWarning: Passing `log_likelihood` via `idata_kwargs` is deprecated and will be removed in future versions. Call `pm.compute_log_likelihood(idata)` instead.
  return _sample_return(
Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 22 seconds.


In [7]:
lagged_fff_model, lagged_fff_output, lagged_fff_resid = bayesian_model_unstandardized(pa_gs_lagged_vars, 'first_fall_frost_doy', all_covs_list)

                                                            Grad                                                  
  Progress               Draw        Divergen…   Step size   evals       Speed                Elapsed    Remaini…  
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.123       63          163.83 draws/s       0:00:12    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.181       31          126.52 draws/s       0:00:15    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.143       31          155.78 draws/s       0:00:12    0:00:00   
  ━━━━━━━━━━━━━━━━━━━━   2000        0           0.110       31          132.21 draws/s       0:00:15    0:00:00

/home/reu/.venv/lib/python3.14/site-packages/pymc/sampling/mcmc.py:1163: FutureWarning: Passing `log_likelihood` via `idata_kwargs` is deprecated and will be removed in future versions. Call `pm.compute_log_likelihood(idata)` instead.
  return _sample_return(
Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 22 seconds.


In [34]:
lagged_gsl_predictions = lagged_model.predict(lagged_output, inplace=False).posterior["mu"].mean(("chain", "draw")).values
lagged_gsl_predictions_series = pd.Series(lagged_gsl_predictions)

all_covs_w_gsl = all_covs_list + ['growing_season_length', 'station_id', 'year']
pa_gs_lagged_select_vars = remove_nulls_simple(pa_gs_lagged_vars, all_covs_w_gsl)
#pa_gs_lagged_no_na_gsl = pa_gs_lagged_no_na['growing_season_length']

pa_gs_lagged_select_vars[['growing_season_length', 'station_id', 'year']].head(5)

pa_gs_lagged_w_preds = pd.concat([pa_gs_lagged_select_vars, lagged_gsl_predictions_series], axis=1)
pa_gs_lagged_w_preds.columns= [*pa_gs_lagged_w_preds.columns[:-1], 'gsl_preds']

pa_gs_gsl_obs = pa_gs_lagged_select_vars.pivot(index='year', columns='station_id', values='growing_season_length')
pa_gs_gsl_preds = pa_gs_lagged_w_preds.pivot(index='year', columns='station_id', values = 'gsl_preds')

lagged_full_variogram_score = sr.variogram_score(pa_gs_gsl_preds, pa_gs_gsl_obs)

ValueError: Forecasts shape (64, 176) and observations shape (64, 176) are not compatible for broadcasting!